# Pulse 2026: Methodology Review — Assumptions, Evidence, and What They Cost

This is the audit notebook in the analysis sequence. It does not introduce a second sample-size
answer. It stress-tests the criterion and implementation from `03-sample_size_reliability.ipynb`,
then records which conclusions survive the assumptions and which remain lower bounds.

**Purpose of this notebook.** It walks through the reasoning behind "how many measurements does a
country need before its IQB score is reliable?" one step at a time, and states every assumption the
answer rests on. Each assumption is labelled **A1–A8**, tested where it can be tested, and priced —
how much the headline number moves if it is wrong.

It is a companion to `03-sample_size_reliability.ipynb`, which does the full analysis. This one is
built to be read and argued with.

---

## The claim, in one paragraph

A country's IQB score is computed from a finite sample of M-Lab measurements. IQB turns out to be
ten weighted pass/fail threshold tests, so the score can only be wrong if one of those tests lands
on the wrong side of its threshold. Whether it does depends on a single quantity — how far that
country's pass rate sits from the cut-off the reporting percentile implies. That makes the required
sample size computable in closed form for every (country, percentile level) pair. The median
country needs about **560** measurements a month; the spread across countries is seven orders of
magnitude, and nine countries cannot be determined at any sample size.

---

## The parts I am least sure about

| | question | section |
|---|---|---|
| 1 | **Measurements are treated as independent.** I know this is false. I cannot quantify how false from monthly aggregates. Every figure is therefore a minimum. Is knowing the direction of the error enough to keep the conclusions usable? | §5, §10 |
| 2 | **The pass rate is reconstructed by interpolating between nine percentile knots.** Is the bracket convention defensible, or does the ambiguity invalidate per-country numbers? | §6 |
| 3 | **The margin is estimated from 24 months pooled**, but the score published covers one month. Is that the right target, or should everything be monthly? | §7 |
| 4 | **The bar is ±0.05 at 95%.** Is that the right definition of "reliable" for this index? | §8 |

Sections §2–§4 are the parts I am confident about and can prove. §5–§8 are the judgement calls.
§9 is the answer. §10 is what I already know is wrong.


## 1. Setup

The machinery is identical to notebook 03 — same functions, same constants — reproduced here so this notebook stands alone and can be run in front of you.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq
from scipy import stats

from iqb import IQBCalculator
from iqb.calculator.config import IQB_CONFIG

calculator = IQBCalculator()

DATA_DIR = Path("./data/v1")
FIG_DIR = Path("./analysis_outputs/review_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
QLEV = np.array(PERCENTILES) / 100.0
METRICS = ["download", "upload", "latency", "loss"]

Q_HEADLINE = 0.50      # the reporting level everything is quoted at unless stated
TOL        = 0.05      # tolerated absolute error on the 0-1 IQB scale        <- assumption A7
CONF       = 0.95      # confidence                                          <- assumption A7
TINY       = 1e-9

RAW_KEY = {"download": "download_throughput_mbps", "upload": "upload_throughput_mbps",
           "latency": "latency_ms", "loss": "packet_loss"}
HIGHER_BETTER = {"download": True, "upload": True, "latency": False, "loss": False}
UNIT = {"download": "Mbps", "upload": "Mbps", "latency": "ms", "loss": "frac"}
# Latency and loss percentiles are reported in the DOWNLOADS table, so they rest on the
# download-side count; only the upload tests rest on the upload-side one.
BACKED_BY = {"download": "down", "upload": "up", "latency": "down", "loss": "down"}

INK, ACCENT, WARN, MUTED = "#1a202c", "#2b6cb0", "#c53030", "#718096"
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linestyle": "--", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


def save(fig, name):
    p = FIG_DIR / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"  saved {p}")


def country_name(cc):
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


print(f"tolerance +/-{TOL} | confidence {CONF:.0%} | headline level p{int(Q_HEADLINE*100)}")
print(f"figures will be written to {FIG_DIR}/")

In [ ]:
def discover_periods(cache_root: Path):
    out = []
    for period_dir in sorted(cache_root.iterdir()):
        if not period_dir.is_dir():
            continue
        for end_dir in sorted(period_dir.iterdir()):
            dl = end_dir / "downloads_by_country" / "data.parquet"
            ul = end_dir / "uploads_by_country" / "data.parquet"
            if dl.exists() and ul.exists():
                out.append((f"{period_dir.name[:4]}-{period_dir.name[4:6]}", dl, ul))
    return out


def load_all(cache_root: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for month, dl_path, ul_path in discover_periods(cache_root):
        if dl_path.stat().st_size < 1000 or ul_path.stat().st_size < 1000:
            continue
        dl = pq.read_table(dl_path).to_pandas()
        ul = pq.read_table(ul_path).to_pandas()
        m = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))
        m["n_down"] = m["sample_count_down"].astype(float)
        m["n_up"] = m["sample_count_up"].astype(float)
        m["up_ratio"] = m["n_up"] / m["n_down"].clip(lower=1)
        m["month"] = month
        frames.append(m)
    return pd.concat(frames, ignore_index=True)


df = load_all()
latest = df[df.month == df.month.max()].reset_index(drop=True)
cc_arr = df.country_code.to_numpy()
n_arr = df.n_down.to_numpy(float)
nup_arr = df.n_up.to_numpy(float)
ratio_arr = df.up_ratio.to_numpy(float)

print(f"{len(df):,} country-months  ({df.country_code.nunique()} countries x "
      f"{df.month.nunique()} months, {df.month.min()} .. {df.month.max()})")
print(f"download-side measurements per country-month: median {df.n_down.median():,.0f}  |  "
      f"P10 {df.n_down.quantile(.1):,.0f}  |  P90 {df.n_down.quantile(.9):,.0f}")

## 2. Step One — IQB Is Ten Weighted Yes/No Tests

**Claim.** The IQB configuration — 6 use cases, each with weighted network requirements — collapses
to exactly ten *(metric, threshold)* pass/fail tests with weights summing to 1:

$$\mathrm{IQB} \;=\; \sum_{k=1}^{10} w_k \cdot
\mathbf{1}\{\text{metric}_k \text{ passes threshold}_k\}, \qquad \sum_k w_k = 1$$

There are 24 use-case × requirement combinations but only 10 distinct tests, because use cases share
thresholds.

**Why it matters.** It means the score cannot move a little — it can only jump. An estimated IQB differs
from the true one by exactly the total weight of the tests that came out wrong. "How many
measurements do I need?" becomes "how many do I need to get ten yes/no answers right?", which is a
question about proportions rather than about four continuous distributions.

**This is not an assumption.** It is a fact about the config, and the cell below proves it by
scoring 400 country-months nine ways each and comparing against the production `IQBCalculator`.
If the printed error is not at machine precision, everything downstream is void.

In [ ]:
def derive_atoms(cfg=IQB_CONFIG):
    ucw_total = sum(uc["w"] for uc in cfg["use cases"].values())
    agg = {}
    for uc in cfg["use cases"].values():
        nrw_total = sum(nr["w"] for nr in uc["network requirements"].values())
        for nr_name, nr in uc["network requirements"].items():
            short = next(k for k, v in RAW_KEY.items() if v == nr_name)
            key = (short, nr["threshold min"])
            agg[key] = agg.get(key, 0.0) + (uc["w"] / ucw_total) * (nr["w"] / nrw_total)
    out = {m: [] for m in METRICS}
    for (m, thr), w in agg.items():
        out[m].append((thr, w))
    for m in out:
        out[m].sort(key=lambda t: t[0], reverse=not HIGHER_BETTER[m])
    return out


ATOMS = derive_atoms()
ATOM_LABELS = [f"{m} {'>' if HIGHER_BETTER[m] else '<'} {thr}"
               for m in METRICS for thr, _ in ATOMS[m]]
ATOM_W = np.array([w for m in METRICS for _, w in ATOMS[m]])
HEAVY = ATOM_W > TOL


def iqb_from_atoms(vals: dict) -> float:
    return sum(w * ((vals[m] > thr) if HIGHER_BETTER[m] else (vals[m] < thr))
               for m in METRICS for thr, w in ATOMS[m])


check = df.sample(400, random_state=0)
worst = max(
    abs(iqb_from_atoms({m: float(r[f"{m}_p{p}"]) for m in METRICS})
        - calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[m]: float(r[f"{m}_p{p}"]) for m in METRICS}}))
    for _, r in check.iterrows() for p in PERCENTILES)

print(f"{'the ten tests':24s}{'weight':>8s}   share of the scale")
for lbl, w in zip(ATOM_LABELS, ATOM_W):
    print(f"{lbl:24s}{w:8.4f}   {'#' * int(round(w * 100))}")
print(f"{'TOTAL':24s}{ATOM_W.sum():8.4f}")
print(f"\nVERIFIED: max |flattened - IQBCalculator| over {len(check)} country-months "
      f"x 9 levels = {worst:.2e}")
print(f"{HEAVY.sum()} of the 10 tests weigh more than the +/-{TOL} tolerance, so flipping any one "
      f"of them alone\nbreaks the bar. That is why the analysis is about individual tests, not "
      f"about an average.")

## 3. Step Two — A Percentile Level Is a Cut-Off on a Count

**Claim.** Evaluating IQB at percentile level *q* asks whether the *q*-quantile of each metric
passes each threshold. Write $\pi$ for the fraction of a country's measurements that beat a
threshold. Then the *q*-quantile passes that test exactly when

$$q \;>\; 1 - \pi$$

Read that out loud and it says something plainer: **evaluating at p50 is asking "do more than 50%
of measurements pass?"; at p75, "do more than 25% pass?"; at p99, "do more than 1% pass?"**

So each of the nine reporting levels converts into a **cut-off on a count of measurements**, and
all nine cut-offs are applied to the *same* pass count. A country is hard to score at a level only
if one of its ten pass counts happens to land near that level's cut-off.

> ### Assumption A2 — the percentile index is a quality rank
> For download this is obvious: `download_p25` is the 25th-percentile speed. For latency and loss
> the pipeline has already flipped direction, so `latency_p1` is the *worst* latency in a country
> and `latency_p99` the best. The single rule above only works if that holds for all four metrics.
> **Tested below** by checking that quality is monotone in the percentile index across every
> country-month. If it fails anywhere, latency and loss are being scored backwards.

In [ ]:
def to_quality(metric, v):
    """Map raw values into a space where higher is always better and passing means y > t."""
    v = np.asarray(v, dtype=float)
    if metric in ("download", "upload"):
        return np.log(np.maximum(v, TINY))       # log: throughput spans decades
    if metric == "latency":
        return -np.log(np.maximum(v, TINY))
    return -v                                     # loss: linear, it has a point mass at 0


print("A2 — quality must increase with the percentile index:")
ok = True
for m in METRICS:
    Y = to_quality(m, df[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
    bad = int((np.diff(Y, axis=1) < -1e-12).any(axis=1).sum())
    ok &= bad == 0
    print(f"   {m:9s} country-months where it decreases: {bad} / {len(df)}")
print(f"\n{'VERIFIED' if ok else 'FAILED'} — A2 holds for all four metrics.")

### 3b. The Picture That Makes This Concrete

One country, one threshold, one pass count — and the nine cut-offs the nine reporting levels apply
to it. Everything about the country is fixed across all nine rows. **Only the cut-off moves.**

This is the figure I would use to explain the whole analysis to someone in thirty seconds.

In [ ]:
# Pick a well-measured country where exactly one level is a 50/50 result and the rest are certain.
cands = []
for _, row in latest.iterrows():
    for m in ("download", "upload"):
        n = float(row.n_up if BACKED_BY[m] == "up" else row.n_down)
        if not (10_000 <= n <= 200_000):
            continue
        for thr, w in ATOMS[m]:
            if w <= TOL:
                continue
            knots = np.array([row[f"{m}_p{p}"] for p in PERCENTILES], float)
            if not (knots[0] < thr < knots[-1]):
                continue
            j = int(np.searchsorted(knots, thr))
            frac = ((np.log(thr) - np.log(knots[j - 1]))
                    / (np.log(knots[j]) - np.log(knots[j - 1])))
            pi = 1 - (QLEV[j - 1] + frac * (QLEV[j] - QLEV[j - 1]))
            z = np.abs(pi * n - (1 - QLEV) * n) / np.sqrt(n * pi * (1 - pi))
            if (z < 1).sum() == 1 and PERCENTILES[int(z.argmin())] != 50:
                cands.append((np.sort(z)[1], row.country_code, m, thr, pi, n))

_, EX_CC, EX_M, EX_T, pi_ex, n_ex = max(cands)
passing = pi_ex * n_ex
spr = np.sqrt(n_ex * pi_ex * (1 - pi_ex))
amb = [p for p in PERCENTILES if abs(passing - (1 - p / 100) * n_ex) / spr < 1][0]

print(f"{country_name(EX_CC)}: {int(n_ex):,} {EX_M} measurements, of which {passing:,.0f} beat "
      f"{EX_T:g} {UNIT[EX_M]}.")
print(f"Re-run the month with a different set of users and that count would move by "
      f"about +/-{spr:,.0f}.\n")
print(f"{'level':>7s}{'is really asking':>26s}{'cut-off':>10s}{'gap to it':>12s}{'verdict':>12s}")
for lvl in PERCENTILES:
    gap = passing - (1 - lvl / 100) * n_ex
    z = abs(gap) / spr
    print(f"{'p' + str(lvl):>7s}{f'do >{100-lvl}% pass?':>26s}{(1-lvl/100)*n_ex:>10,.0f}"
          f"{gap:>+12,.0f}{('certain' if z > 3 else '50/50' if z < 1 else 'unclear'):>12s}")
print(f"\nSame measurements, same threshold, same spread. Only the cut-off moves - and at p{amb} it "
      f"falls\nalmost exactly on the pass count, so that one level is a 50/50 result even with "
      f"{int(n_ex):,} measurements.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), constrained_layout=True)

ax = axes[0]
tr = ax.get_xaxis_transform()
for lvl in PERCENTILES:
    cut, hot = (1 - lvl / 100) * n_ex, lvl == amb
    ax.axvline(cut, color=WARN if hot else "#4a5568", lw=1.8 if hot else 1)
    ax.text(cut, 0.9, f"p{lvl}", ha="center", fontsize=8, transform=tr,
            color=WARN if hot else "#4a5568", fontweight="bold" if hot else "normal")
ax.axvline(passing, color=WARN, lw=2.6)
ax.text(passing, 0.32, f" {passing:,.0f} measurements\n pass the threshold", fontsize=8.5,
        transform=tr, color=WARN, va="center")
ax.set_xlim(-n_ex * 0.03, n_ex * 1.03)
ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("number of measurements")
ax.set_title(f"Nine reporting levels are nine cut-offs on one count\n"
             f"{country_name(EX_CC)}, {EX_M} > {EX_T:g} {UNIT[EX_M]}, n = {int(n_ex):,}",
             fontsize=10)

zs = {lvl: ((1 - lvl / 100) * n_ex - passing) / spr for lvl in PERCENTILES}
ax = axes[1]
tr = ax.get_xaxis_transform()
ax.axvspan(-1.96, 1.96, color=WARN, alpha=0.16)
ax.text(0, 0.10, "where the count could\nplausibly have landed", ha="center", fontsize=8.5,
        transform=tr, color=WARN)
ax.axvline(0, color=WARN, lw=2.6)
for lvl, z in zs.items():
    if abs(z) < 11:
        ax.axvline(z, color=WARN, lw=1.8)
        ax.text(z, 0.9, f"p{lvl}", ha="center", fontsize=9, transform=tr,
                color=WARN, fontweight="bold")
ax.annotate(f"every other level is\n{min(abs(z) for l, z in zs.items() if l != amb):.0f}+ "
            f"standard deviations away,\noff this chart", xy=(0.97, 0.55), xycoords="axes fraction",
            ha="right", fontsize=8.5, color=MUTED)
ax.set_xlim(-11, 11)
ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("distance from the pass count, in standard deviations")
ax.set_title(f"Zoomed in: only p{amb}'s cut-off falls inside\nthe range the count could land in",
             fontsize=10)
save(fig, "fig1_nine_cutoffs")
plt.show()

## 4. Step Three — How Far the Count Can Wander

**Claim.** Each measurement either passes a threshold or does not, so the number that pass out of
$n$ is $\mathrm{Binomial}(n, \pi)$, with standard deviation — the **spread** —

$$\sqrt{n\,\pi(1-\pi)}$$

The level's cut-off sits at $(1-q)n$ measurements, so the gap between the pass count and the cut-off
is $\delta n$, writing $\delta = |\pi - (1-q)|$ for the **margin**. Demanding the gap beat the
spread by $z$ standard deviations gives the ordinary sample size for a proportion:

$$n \;>\; \frac{\pi(1-\pi)\,z^2}{\delta^2}$$

**The consequence worth internalising.** Note what does *not* appear: the metric's units, its
variance, how heavy-tailed it is. A country with wildly dispersed download speeds is no harder to
score than a stable one. **Only the margin matters** — and because it enters squared, halving the
margin quadruples the requirement.

### The exact version actually used

The formula above is a normal approximation, fine at $n = 20{,}000$ and poor at $n = 20$. Much of
this panel is small, so the computation uses the exact order-statistic result instead: for $n$
i.i.d. measurements the sample $q$-quantile is the $r$-th order statistic, $r = \lceil qn \rceil$,
and its position on the true CDF is Beta distributed,

$$U = F(\hat{Q}_q) \sim \mathrm{Beta}(r,\; n - r + 1)$$

so test $k$ looks like a pass exactly when $U > 1 - \pi_k$.

> ### Assumption A3 — measurements are independent and identically distributed
> **This is the assumption everything else in the analysis depends on, and it is the one I am least able to
> defend.** Everything above treats a country's measurements as *n* independent draws. M-Lab tests
> are self-selected and cluster by user, network and time of day. §10 quantifies how bad the
> clustering looks. What it means: every number in this notebook is a **lower bound** on what is
> really needed.
>
> **What is genuinely verified below** is the mathematics conditional on A3: that the Beta form is
> exact (including for packet loss, which has a point mass at zero and so is not continuous), and
> that it reproduces the production calculator.

In [ ]:
BANDS = {m: np.concatenate([[0.0], np.cumsum([w for _, w in ATOMS[m]])]) for m in METRICS}
KS = [len(ATOMS[m]) for m in METRICS]
LATTICE = sum(g.ravel() for g in np.meshgrid(*[BANDS[m] for m in METRICS], indexing="ij"))


def sampling_dist(C, q, n, ratio=1.0):
    """[rows x 144] exact distribution of the IQB estimate. C = cut points (1 - pass rate).
    n is the download-side count; upload tests are evaluated at n * ratio."""
    rows = len(C)
    n_dn = np.full(rows, max(int(n), 1))
    n_up = np.maximum(np.round(n_dn * np.broadcast_to(np.asarray(ratio, float), (rows,))), 1)
    parts, off = [], 0
    for m, K in zip(METRICS, KS):
        n_m = (n_up if BACKED_BY[m] == "up" else n_dn).astype(np.int64)
        r = np.clip(np.ceil(q * n_m), 1, n_m).astype(np.int64)
        s = stats.beta.sf(C[:, off:off + K], r[:, None], (n_m - r + 1)[:, None])
        off += K
        p = np.empty((rows, K + 1))
        p[:, 0] = 1.0 - s[:, 0]
        for j in range(1, K):
            p[:, j] = s[:, j - 1] - s[:, j]
        p[:, K] = s[:, K - 1]
        parts.append(np.clip(p, 0.0, None))
    J = parts[0]
    for p in parts[1:]:
        J = (J[:, :, None] * p[:, None, :]).reshape(rows, -1)
    return J


def true_scores(P, q):
    t = np.zeros(len(P))
    for k in range(P.shape[1]):
        t += ATOM_W[k] * (q > 1.0 - P[:, k])
    return t


def p_within(C, t, q, n, ratio=1.0, tol=TOL):
    J = sampling_dist(C, q, n, ratio)
    return (J * (np.abs(LATTICE[None, :] - t[:, None]) <= tol + 1e-12)).sum(axis=1)


NGRID = np.unique(np.round(np.logspace(0, 8.3, 170)).astype(int))


def required_n(C, t, q, ratio=1.0, tol=TOL, conf=CONF):
    """Smallest n at which the bar is met THERE AND AT EVERY LARGER n.
    The trailing condition is not padding: r = ceil(qn) makes r/n jump around q, so reliability
    can genuinely worsen when one more measurement is added. Taking the first crossing alone
    would report a sawtooth artefact as an answer."""
    ok = np.array([p_within(C, t, q, int(n), ratio, tol) >= conf for n in NGRID])
    allok = np.empty_like(ok)
    acc = np.ones(len(C), bool)
    for i in range(len(NGRID) - 1, -1, -1):
        acc = acc & ok[i]
        allok[i] = acc
    return np.where(allok.any(axis=0), NGRID[np.argmax(allok, axis=0)], np.inf).astype(float)


print(f"IQB estimates live on a {len(LATTICE)}-point lattice, "
      f"{LATTICE.min():.3f} .. {LATTICE.max():.3f}")
print("Within a metric the thresholds are nested (passing 25 Mbps implies passing 10), so all")
print("thresholds on a metric are compared against ONE draw of U. That is exact, not an "
      "approximation.")

In [ ]:
# --- A3 check 1: is the Beta form exact when the distribution has an atom? (loss = 0 is common)
rng = np.random.default_rng(1)
print("Beta closed form vs simulation, for a distribution with a 60% point mass at zero:")
print(f"{'level':>8s}{'n':>7s}{'simulated':>12s}{'closed form':>13s}")
thr, F = 0.01, 0.60 + 0.40 * (0.01 / 0.05)
for q, n in [(0.50, 50), (0.50, 501), (0.25, 101), (0.90, 200)]:
    r = int(np.clip(np.ceil(q * n), 1, n))
    draws = np.where(rng.random((40000, n)) < 0.60, 0.0, rng.random((40000, n)) * 0.05)
    sim = float(np.mean(np.sort(draws, axis=1)[:, r - 1] < thr))
    print(f"{'p' + str(int(q*100)):>8s}{n:>7d}{sim:>12.4f}{stats.beta.cdf(F, r, n - r + 1):>13.4f}")
print("\nVERIFIED: the order-statistic result does not require a continuous distribution.")

## 5. Step Four — Getting the Pass Rate Out of Nine Percentiles

Everything so far needs $\pi$, the fraction of a country's measurements that pass a threshold. **The
parquet does not report it.** It reports nine percentiles per metric, so $\pi$ has to be recovered
by working out where the threshold falls between those nine knots.

> ### Assumption A4 — nine percentile knots are enough to locate a threshold
> Between two adjacent distinct knot values the definition of a quantile *brackets* the CDF:
> $F(t) \in [\,q_{\max}(\text{left}),\; q_{\min}(\text{right})\,)$. Within that bracket I
> interpolate linearly. **Nine knots genuinely do not determine $F$ between them** — this is a
> choice, not a derivation, and §10 prices three alternatives against it.
>
> Two details that are not arbitrary. Interpolation happens in **quality space** (log for
> throughput, which spans decades; linear for loss, which has a point mass at zero that a log would
> distort). And a threshold falling **outside** the reported p1–p99 range yields only a bound, which
> is pushed just past the edge rather than set exactly at it — returning exactly 0.01 would place the cut
> point precisely at the p99 reporting level and create a permanently unpredictable test there.

In [ ]:
def pass_prob(y_knots, t):
    """P(one measurement passes) = 1 - F(t), read off the nine knots."""
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if t <= uy[0]:
        return 0.995                                   # F <= 0.01 -> pi >= 0.99 (bound only)
    if t >= uy[-1]:
        return 0.005                                   # F >= 0.99 -> pi <= 0.01 (bound only)
    i = np.searchsorted(uy, t) - 1
    F = q_hi[i] + (q_lo[i + 1] - q_hi[i]) * (t - uy[i]) / (uy[i + 1] - uy[i])
    return float(np.clip(1.0 - F, 0.005, 0.995))


def pass_prob_matrix(frame) -> np.ndarray:
    cols = []
    for m in METRICS:
        Y = to_quality(m, frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            cols.append(np.array([pass_prob(Y[i], t) for i in range(len(Y))]))
    return np.column_stack(cols)


def threshold_in_range(frame) -> np.ndarray:
    out = []
    for m in METRICS:
        Y = to_quality(m, frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            out.append((Y.min(axis=1) < t) & (t < Y.max(axis=1)))
    return np.column_stack(out)


P_all = pass_prob_matrix(df)
in_range = threshold_in_range(df)

print(f"pass-rate matrix: {P_all.shape}  (country-months x tests)\n")
print("how often each threshold falls OUTSIDE the reported p1-p99 range, so pi is only bounded:")
for k, lbl in enumerate(ATOM_LABELS):
    bar = "#" * int(round((~in_range[:, k]).mean() * 40))
    print(f"   {lbl:16s}{(~in_range[:, k]).mean():7.1%}  {bar}")
print(f"\noverall: {(~in_range).mean():.1%} of country-month-threshold pairs.")
print("The two loss tests - which set the requirement for about half the panel - are almost always")
print("in range, so the bounded cases are mostly the light latency tests. That limits the damage.")

## 6. Step Five — Estimating the Margin Without Circularity

There is an obvious circularity to deal with. The margin has to be estimated from the same
percentile data whose reliability is in question, and a country with three measurements gives a
useless estimate of its own pass rates.

**Pooling all 24 months fixes it, and does so exactly.** Notebook 02 establishes that monthly
distributions must be combined by averaging **CDFs**, since the mean of two p50s is not the p50 of
the pooled data. A pass rate is that same CDF evaluated at a *fixed* $x$ — the threshold — so

$$\pi^{\text{pooled}}_k \;=\; 1 - F_{\text{merged}}(t_k)
\;=\; \sum_i w_i\bigl(1 - F_i(t_k)\bigr) \;=\; \sum_i w_i \,\pi_{k,i}$$

is an **identity, not an approximation**. What cannot be averaged is a *quantile* — a position on
the vertical axis that moves when the distribution changes shape. A pass rate sits on the horizontal
axis at a pinned $x$, and those add linearly.

> ### Assumption A5 — the 24-month pooled distribution is the right target
> This makes every requirement below the sample size at which the reported score matches the score
> of the country's **pooled** distribution. That is the right target for capacity planning and the
> wrong one for auditing a single month. §10 recomputes everything month by month and prices the
> difference. **This is the assumption I am least sure is the right modelling choice** (as opposed
> to A3, which I am sure is wrong and merely cannot fix).

In [ ]:
codes, P_rows, n_med, n_months, ratio_rows = [], [], [], [], []
for c in pd.unique(cc_arr):
    m = cc_arr == c
    w = n_arr[m]
    w = w / w.sum() if w.sum() > 0 else np.full(m.sum(), 1 / m.sum())
    codes.append(c)
    P_rows.append((P_all[m] * w[:, None]).sum(axis=0))     # the identity above
    n_med.append(np.median(n_arr[m]))
    n_months.append(int(m.sum()))
    ratio_rows.append(nup_arr[m].sum() / max(n_arr[m].sum(), 1.0))

P_c = np.array(P_rows)
C_c = 1 - P_c
n_med = np.array(n_med)
ratio_c = np.array(ratio_rows)
t_c = true_scores(P_c, Q_HEADLINE)

margin_c = np.abs(P_c - (1 - Q_HEADLINE))
kbind = np.where(HEAVY, margin_c, np.inf).argmin(axis=1)
margin_heavy = np.where(HEAVY, margin_c, np.inf).min(axis=1)

print(f"pooled to {len(codes)} countries.\n")
print("the margin - distance from the nearest consequential cut-off, on the pass-rate scale:")
for lab, qq in [("P10 (most exposed)", .10), ("P25", .25), ("median", .50),
                ("P75", .75), ("P90 (most secure)", .90)]:
    print(f"   {lab:22s}{np.quantile(margin_heavy, qq):8.4f}")
print(f"\nwhich test sets the requirement, across the {len(codes)} countries:")
print(pd.Series(np.array(ATOM_LABELS)[kbind]).value_counts().to_string())

## 7. Validating the Whole Chain Against the Production Calculator

Steps one to five are now a closed-form claim: given a country's nine percentile knots and a sample
size, here is the exact distribution of the IQB score you would report. That claim is checkable.

Below, synthetic measurements are drawn from each country's reconstructed distribution, the sample
percentile is taken, and the resulting four metric values are scored with `IQBCalculator` — the same
code path production uses. Monte-Carlo and closed form should agree to within simulation error.

> **What this does and does not establish.** Both sides draw from the *same* reconstructed CDF, so
> this validates the Beta algebra, the 144-point lattice and the config flattening — **not** A4, the
> reconstruction itself. No within-month replicate exists in a monthly pre-aggregate, so A4 cannot
> be validated from this data at all. That is the strongest argument for asking the pipeline to emit
> the ten pass rates directly.

In [ ]:
rng = np.random.default_rng(0)


def inv_quality_cdf(y_knots, u):
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if u <= q_lo[0]:
        return uy[0]
    if u >= q_hi[-1]:
        return uy[-1]
    i = int(np.clip(np.searchsorted(q_hi, u, side="right") - 1, 0, len(uy) - 2))
    if u <= q_hi[i]:
        return uy[i]
    span = q_lo[i + 1] - q_hi[i]
    return uy[i] + (0.0 if span <= 0 else (u - q_hi[i]) / span) * (uy[i + 1] - uy[i])


def from_quality(metric, y):
    if metric in ("download", "upload"):
        return float(np.exp(y))
    if metric == "latency":
        return float(np.exp(-y))
    return float(-y)


def monte_carlo(row, q, n, ratio, reps=2000):
    n_m = {m: max(int(round(n * ratio)) if BACKED_BY[m] == "up" else int(n), 1) for m in METRICS}
    r_m = {m: int(np.clip(np.ceil(q * v), 1, v)) for m, v in n_m.items()}
    knots = {m: to_quality(m, [row[f"{m}_p{p}"] for p in PERCENTILES]) for m in METRICS}
    out = np.empty(reps)
    for i in range(reps):
        vals = {m: from_quality(m, inv_quality_cdf(
                    knots[m], np.sort(rng.random(n_m[m]))[r_m[m] - 1])) for m in METRICS}
        out[i] = calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[k]: v for k, v in vals.items()}})
    return out


P_lat = pass_prob_matrix(latest)
marg_lat = np.where(HEAVY, np.abs(P_lat - (1 - Q_HEADLINE)), np.inf).min(axis=1)
ws = np.where(latest.n_down.to_numpy() >= 1000)[0]
TIGHT = latest.country_code.iloc[ws[np.argmin(marg_lat[ws])]]     # hardest case available

print(f"tightest-margin well-sampled country this month: {TIGHT} ({country_name(TIGHT)}) - "
      f"the case most likely to expose an error\n")
print(f"{'country':>8s}{'n':>7s} | {'true':>6s}{'MC mean':>9s}{'exact':>8s} | "
      f"{'MC sd':>7s}{'exact':>8s} | {'MC P(ok)':>10s}{'exact':>8s}")
for cc in ["US", "BR", "IN", "NG", TIGHT]:
    sub = latest[latest.country_code == cc]
    if sub.empty:
        continue
    Ps = pass_prob_matrix(sub)
    C, t = 1 - Ps, true_scores(Ps, Q_HEADLINE)
    rt = float(sub.up_ratio.iloc[0])
    for n in (100, 1000):
        J = sampling_dist(C, Q_HEADLINE, n, rt)
        mean = float((J * LATTICE).sum())
        sd = float(np.sqrt((J * (LATTICE[None, :] - mean) ** 2).sum()))
        pok = float(p_within(C, t, Q_HEADLINE, n, rt)[0])
        sim = monte_carlo(sub.iloc[0], Q_HEADLINE, n, rt)
        print(f"{cc:>8s}{n:>7d} | {t[0]:6.3f}{sim.mean():>9.3f}{mean:>8.3f} | "
              f"{sim.std():>7.3f}{sd:>8.3f} | "
              f"{np.mean(np.abs(sim - t[0]) <= TOL + 1e-12):>10.3f}{pok:>8.3f}")
print("\nVERIFIED: closed form and Monte Carlo agree to within simulation error (+/-0.01 at "
      "2,000 reps).")

## 8. What the Margin Gives You

Before the answer, the shape of it. The requirement depends on the margin **squared**, so the
spread across countries is not a nuisance — it is the entire story. This is the figure I would put
in front of anyone who asks "why can't you just pick a cutoff?"

In [ ]:
# The requirement for every country, used here and in section 9.
req_base = required_n(C_c, t_c, Q_HEADLINE, ratio_c)

z = stats.norm.ppf(1 - (1 - CONF) / 2)
d_grid = np.logspace(-4, np.log10(0.5), 300)

fig, ax = plt.subplots(figsize=(8.4, 5))
ax.plot(d_grid, 0.25 * (z / d_grid) ** 2, color=ACCENT, lw=2.2, zorder=2,
        label="textbook formula for a proportion")
ax.set_xscale("log")
ax.set_yscale("log")

label = {"TM": "Turkmenistan", "GB": "United Kingdom", "ID": "Indonesia",
         "IN": "India", "MX": "Mexico", "CA": "Canada", "HU": "Hungary"}
for cc, nm in label.items():
    if cc not in codes:
        continue
    i = codes.index(cc)
    d, n_exact = margin_heavy[i], req_base[i]
    if not np.isfinite(n_exact):
        continue
    ax.scatter([d], [n_exact], s=46, color=WARN, zorder=4, edgecolor="white", linewidth=0.8)
    ax.annotate(f" {nm}", (d, n_exact), fontsize=8.5, color=INK, va="center", zorder=5)

ax.axvspan(d_grid[0], 0.002, color=WARN, alpha=0.07, zorder=1)
ax.text(3.2e-4, 3e6, "no realistic\nsample size", fontsize=8.5, color=WARN, ha="center")
ax.scatter([], [], s=46, color=WARN, edgecolor="white", linewidth=0.8,
           label="exact requirement, real countries")
ax.set_xlim(7e-5, 1.6)
ax.legend(fontsize=8.5, loc="lower left", framealpha=0.95)
ax.set_xlabel("margin — distance from the nearest consequential cut-off (pass-rate scale)")
ax.set_ylabel("measurements needed")
ax.set_title("Halve the distance to a threshold, quadruple the data you need", fontsize=11)
save(fig, "fig2_margin_to_sample_size")
plt.show()

print("The dots are each country's exact requirement; the line is the textbook approximation.")
print("They part company only at the very small end, where the normal approximation stops holding.")
print("\nA country's size, its population, and how variable its internet is do not appear anywhere")
print("in that relationship. Only its distance from a threshold does - which is an accident of")
print("where its performance happens to fall relative to numbers written into the IQB config.")

## 9. The Answer

In [ ]:
enough = n_med >= req_base

fin = pd.Series(req_base).replace(np.inf, np.nan)
print(f"=== download-side measurements needed, p{int(Q_HEADLINE*100)}, "
      f"+/-{TOL} at {CONF:.0%} ===")
for lab, qq in [("P10", .10), ("P25", .25), ("median", .50), ("P75", .75), ("P90", .90)]:
    print(f"   {lab:8s}{fin.quantile(qq):>12,.0f}")
print(f"   {'range':8s}{fin.min():>12,.0f}  ..{fin.max():>14,.0f}"
      f"   ({int(np.isinf(req_base).sum())} unreachable at any n)")
print(f"\n{enough.sum()} of {len(codes)} countries already collect enough; "
      f"{(~enough).sum()} do not.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
v = np.sort(np.nan_to_num(req_base, posinf=1e9))
ax.plot(v, np.arange(1, len(v) + 1) / len(v), lw=2.2, color=ACCENT)
for g, lab in [(1_000, "n=1,000"), (10_000, "n=10,000")]:
    ax.axvline(g, color=MUTED, ls=":", lw=1)
    ax.text(g * 1.15, 0.04, lab, fontsize=8, color=MUTED)
ax.set_xscale("log")
ax.set_xlim(1, 1e7)
ax.set_xlabel("measurements per month required")
ax.set_ylabel("share of countries covered")
ax.set_title("No single cutoff covers everyone", fontsize=10.5)

ax = axes[1]
ax.scatter(np.maximum(n_med[enough], 1), req_base[enough], s=18, alpha=0.6,
           color="#38a169", label=f"has enough ({enough.sum()})")
ax.scatter(np.maximum(n_med[~enough], 1), np.nan_to_num(req_base[~enough], posinf=1e8),
           s=18, alpha=0.6, color=WARN, label=f"short ({(~enough).sum()})")
ax.plot([1, 1e8], [1, 1e8], color=INK, lw=1, ls="--", label="required = collected")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.7, 1e8)
ax.set_ylim(0.7, 1e9)
ax.set_xlabel("measurements collected per month")
ax.set_ylabel("measurements required")
ax.set_title("Volume does not predict reliability", fontsize=10.5)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
save(fig, "fig3_the_answer")
plt.show()

## 10. Every Assumption, Priced

This is the section to argue with. Each assumption is varied to a defensible alternative and the
**median requirement** recomputed. If an assumption barely moves the number, disagreeing about it
does not matter. If it moves it a lot, it needs to be right.

In [ ]:
def pass_prob_variant(y_knots, t, tie="bracket"):
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    if tie == "bracket":
        return pass_prob(y_knots, t)
    up = np.array([(QLEV[y == v].max() if tie == "top" else QLEV[y == v].min()) for v in uy])
    F = (1.0 if t >= uy[0] else 0.0) if len(uy) == 1 else float(np.interp(t, uy, up))
    return float(np.clip(1.0 - F, 0.005, 0.995))


def variant_matrix(frame, tie="bracket", log_space=True):
    cols = []
    for m in METRICS:
        raw = frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float)
        sign = 1 if m in ("download", "upload") else -1
        Y = to_quality(m, raw) if log_space else raw * sign
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr)) if log_space else float(thr * sign)
            cols.append(np.array([pass_prob_variant(Y[i], t, tie) for i in range(len(Y))]))
    return np.column_stack(cols)


def pool(P):
    rows = []
    for c in codes:
        m = cc_arr == c
        w = n_arr[m]
        w = w / w.sum() if w.sum() > 0 else np.full(m.sum(), 1 / m.sum())
        rows.append((P[m] * w[:, None]).sum(axis=0))
    return np.array(rows)


ledger = {}
ledger["baseline (this analysis)"] = req_base

# --- A4: three alternative reconstructions
A4 = {"A4: nb02 tie convention": dict(tie="top", log_space=True),
      "A4: lower envelope": dict(tie="bottom", log_space=True),
      "A4: linear, not log space": dict(tie="bracket", log_space=False)}
variants_P = {}
for name, kw in A4.items():
    Pv = pool(variant_matrix(df, **kw))
    variants_P[name] = Pv
    ledger[name] = required_n(1 - Pv, true_scores(Pv, Q_HEADLINE), Q_HEADLINE, ratio_c)

# --- A5: target the month being published, not the 24-month pooled distribution
req_month = required_n(1 - P_all, true_scores(P_all, Q_HEADLINE), Q_HEADLINE, ratio_arr)
ledger["A5: monthly target"] = req_month

# --- A7: a looser and a stricter bar
for tol in (0.02, 0.10):
    ledger[f"A7: tolerance +/-{tol}"] = required_n(C_c, t_c, Q_HEADLINE, ratio_c, tol)

print(f"{'variation':34s}{'median':>11s}{'P90':>13s}{'vs baseline':>14s}")
for name, v in ledger.items():
    s = pd.Series(v).replace(np.inf, np.nan)
    r = s.median() / fin.median()
    print(f"{name:34s}{s.median():>11,.0f}{s.quantile(.9):>13,.0f}"
          f"{('baseline' if name.startswith('baseline') else f'{r:.2f}x'):>14s}")

In [ ]:
# --- A6: metric dependence, at the opposite extreme (all ten tests share one CDF position)
def required_n_comonotone(C, t, q, tol=TOL, conf=CONF):
    order = np.argsort(C, axis=1)
    Cs = np.take_along_axis(C, order, axis=1)
    vals = np.concatenate([np.zeros((len(C), 1)), np.cumsum(ATOM_W[order], axis=1)], axis=1)
    ok = []
    for n in NGRID:
        r = int(np.clip(np.ceil(q * n), 1, n))
        sf = stats.beta.sf(Cs, r, n - r + 1)
        pr = np.empty_like(vals)
        pr[:, 0] = 1 - sf[:, 0]
        pr[:, -1] = sf[:, -1]
        pr[:, 1:-1] = sf[:, :-1] - sf[:, 1:]
        ok.append((np.clip(pr, 0, None) *
                   (np.abs(vals - t[:, None]) <= tol + 1e-12)).sum(axis=1) >= conf)
    ok = np.array(ok)
    allok = np.empty_like(ok)
    acc = np.ones(len(C), bool)
    for i in range(len(NGRID) - 1, -1, -1):
        acc = acc & ok[i]
        allok[i] = acc
    return np.where(allok.any(axis=0), NGRID[np.argmax(allok, axis=0)], np.inf).astype(float)


ledger["A6: metrics perfectly correlated"] = required_n_comonotone(C_c, t_c, Q_HEADLINE)

order = ["A7: tolerance +/-0.1", "A6: metrics perfectly correlated", "A4: nb02 tie convention",
         "A4: lower envelope", "A4: linear, not log space", "A7: tolerance +/-0.02",
         "A5: monthly target"]
order = [k for k in order if k in ledger]
base = fin.median()
vals = [pd.Series(ledger[k]).replace(np.inf, np.nan).median() for k in order]

fig, ax = plt.subplots(figsize=(9.6, 4.4))
cols = [WARN if v > base else "#38a169" for v in vals]
ax.barh(range(len(order)), [v - base for v in vals], left=base, color=cols, alpha=0.85, height=0.6)
ax.axvline(base, color=INK, lw=1.6)
ax.text(base, len(order) - 0.3, f"  baseline {base:,.0f}", fontsize=9, color=INK, va="top")
for i, v in enumerate(vals):
    ax.text(v + (18 if v > base else -18), i, f"{v:,.0f}", fontsize=8.5, va="center",
            ha="left" if v > base else "right", color=INK)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlabel("median measurements required")
ax.set_xlim(0, max(vals) * 1.22)
ax.set_title("How much each assumption changes the answer\n"
             "the headline number under each defensible alternative", fontsize=11)
ax.grid(axis="y", visible=False)
save(fig, "fig4_assumption_sensitivity")
plt.show()

print("Read this as: the two things that move the answer are both DEFINITIONAL rather than")
print("empirical - where the bar is set (A7) and what 'true score' means (A5). Every assumption")
print("about the world holds the answer within about 10%: the reconstruction conventions (A4)")
print("barely touch the aggregate, and metric dependence (A6) does not matter at all.")
print("A4 does move individual countries a lot, which the next cell shows.")

In [ ]:
# A4 again, at the level where it DOES matter: individual countries.
P_nb02 = variants_P["A4: nb02 tie convention"]
spread_all = np.abs(P_nb02 - P_c)
spread_bind = spread_all[np.arange(len(P_c)), kbind]
unresolved = spread_bind > margin_heavy
straddle = (((P_c > 1 - Q_HEADLINE) != (P_nb02 > 1 - Q_HEADLINE)) & HEAVY).any(axis=1)

print("A4 at the per-country level (compared on the test that actually sets the requirement):")
print(f"   typical disagreement across the ten tests     "
      f"{np.median(spread_all.max(axis=1)):.4f} on the pass-rate scale")
print(f"   ...on the binding test, where it matters      "
      f"{np.median(spread_bind[spread_bind > 0]):.4f} "
      f"(zero for {int((spread_bind == 0).sum())} countries, where the conventions agree exactly)")
print(f"   countries where it exceeds their own margin  {unresolved.sum():>3d} / {len(codes)}")
print(f"   ...where the two land on opposite sides      {straddle.sum():>3d} / {len(codes)}")
print("\nFor those countries nine knots do not determine the margin, so their requirement is")
print("UNRESOLVED rather than large. More measurement cannot fix it - the ambiguity is in the")
print("nine-number summary, not in the sample. Ten COUNTIF columns in the aggregation would.")

## 11. What I Already Know Is Wrong

Two things. The first is the assumption I cannot fix from this data. The second is a real effect
the analysis found and that nothing in the current pipeline accounts for.

### 11a. A3 — measurements are not independent

Every requirement above assumes a country's measurements are *n* independent draws. They are not:
M-Lab tests are self-selected and cluster heavily by network. The cell below measures how heavily,
using the per-ASN tables that sit alongside the country ones in the same data directory.

**Why this cannot be resolved here.** Quantifying a design effect needs a within-month replicate —
two independent samples of the same country in the same month. A monthly pre-aggregate does not
contain one. So the honest statement is directional: **every figure in this notebook is a lower
bound on the sample size really needed**, and I cannot say by how much.

**The one piece of reassurance.** Decomposing 24 months of country pass rates (in `METHODOLOGY.md`
§8) shows that about 77% of month-to-month movement is change *within* networks and only about 7%
is shifts in which networks got measured. So the composition of the measuring population is at
least fairly stable, even though it is concentrated.

In [ ]:
asn_path = sorted(DATA_DIR.glob("*/*/downloads_by_country_asn/data.parquet"))[-1]
A = pq.read_table(asn_path).to_pandas()
A = A[A.sample_count >= 30]

tot = A.groupby("country_code").sample_count.sum()
share = A.assign(s=A.sample_count / A.country_code.map(tot))
eff = 1 / share.groupby("country_code").s.apply(lambda s: (s ** 2).sum())
top = share.groupby("country_code").s.max()

Y = to_quality("download", A[[f"download_p{p}" for p in PERCENTILES]].to_numpy(float))
t10 = float(to_quality("download", 10.0))
A = A.assign(pi=[pass_prob(Y[i], t10) for i in range(len(Y))])

rows = []
for cc, g in A.groupby("country_code"):
    n = g.sample_count.to_numpy(float)
    if n.sum() < 5000 or len(g) < 4:
        continue
    pbar = float(np.average(g.pi, weights=n))
    rows.append((cc, n.sum(), eff[cc], top[cc],
                 float(np.sqrt(np.average((g.pi - pbar) ** 2, weights=n)))))
K = pd.DataFrame(rows, columns=["cc", "n", "eff_asn", "top_share", "sd_between"])

print(f"latest month, {len(K)} countries with >=5,000 measurements and >=4 ASNs\n")
print(f"   effective number of ASNs (1/HHI)       median {K.eff_asn.median():5.1f}   "
      f"P10 {K.eff_asn.quantile(.1):.1f}   P90 {K.eff_asn.quantile(.9):.1f}")
print(f"   share in the single largest ASN        median {K.top_share.median():5.1%}")
print(f"   spread of the pass rate across ASNs    median {K.sd_between.median():5.3f}   "
      f"P90 {K.sd_between.quantile(.9):.3f}")
print(f"\nFor scale: the median country's binding margin is {np.median(margin_heavy):.4f}. The")
print("variation between a country's own networks is far larger than the margin being resolved.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
v = np.sort(K.eff_asn.to_numpy())
ax.plot(v, np.arange(1, len(v) + 1) / len(v), lw=2.2, color=ACCENT)
ax.axvline(K.eff_asn.median(), color=WARN, ls="--", lw=1.4)
ax.text(K.eff_asn.median() * 1.1, 0.15, f"median {K.eff_asn.median():.1f}", fontsize=9, color=WARN)
ax.set_xscale("log")
ax.set_xlabel("effective number of networks a country's measurements come from (1/HHI)")
ax.set_ylabel("share of countries")
ax.set_title("Measurements concentrate in a handful of networks", fontsize=10.5)

ax = axes[1]
ax.scatter(K.n, K.sd_between, s=20, alpha=0.6, color=ACCENT)
ax.axhline(np.median(margin_heavy), color=WARN, lw=1.6, ls="--")
ax.text(K.n.min() * 1.4, np.median(margin_heavy) * 1.25,
        f"median margin the analysis resolves ({np.median(margin_heavy):.3f})",
        fontsize=8.5, color=WARN)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("measurements collected")
ax.set_ylabel("spread of the pass rate across a country's own networks")
ax.set_title("...and those networks do not agree with each other", fontsize=10.5)
fig.tight_layout()
save(fig, "fig5_independence_problem")
plt.show()

### 11b. The Estimate Is Biased, Not Just Noisy

This is not an assumption — it is a consequence the analysis surfaced, and no part of the current
reporting accounts for it.

A country scoring near 1.0 cannot score any higher: every test that can flip can only flip *downward*.
Near 0 the reverse holds. So the estimate is pulled toward the middle of the lattice, and because
**0 is bad and 1 is good, an under-measured country is published as worse than it is.**

Two properties make this worse than ordinary noise. It does not shrink when the tolerance is
widened — a wider band describes spread around a centre that is in the wrong place. And it does not
average out across countries: every under-sampled country is pushed the same direction at the same
time, so rankings, regional averages and year-on-year comparisons all inherit it.

In [ ]:
print(f"{'n':>9s}{'mean bias':>12s}{'if true > 0.8':>15s}{'if true < 0.4':>15s}"
      f"{'|bias| > tol':>14s}")
grid = (50, 100, 300, 1_000, 3_000, 10_000)
curves = {"all": [], "true > 0.8": [], "true < 0.4": []}
for n in grid:
    b = (sampling_dist(C_c, Q_HEADLINE, n, ratio_c) * LATTICE).sum(axis=1) - t_c
    curves["all"].append(b.mean())
    curves["true > 0.8"].append(b[t_c > 0.8].mean())
    curves["true < 0.4"].append(b[t_c < 0.4].mean())
    print(f"{n:>9,d}{b.mean():>12.3f}{b[t_c > 0.8].mean():>15.3f}{b[t_c < 0.4].mean():>15.3f}"
          f"{(np.abs(b) > TOL).mean():>14.1%}")

bias_now = np.empty(len(codes))
for n in np.unique(np.maximum(n_med, 1).astype(int)):
    idx = np.where(np.maximum(n_med, 1).astype(int) == n)[0]
    bias_now[idx] = (sampling_dist(C_c[idx], Q_HEADLINE, int(n), ratio_c[idx])
                     * LATTICE).sum(axis=1) - t_c[idx]
bad = np.abs(bias_now) > TOL
print(f"\nAt current volumes, {bad.sum()} of {len(codes)} countries carry a bias larger than the "
      f"+/-{TOL} tolerance\nitself. Worst {bias_now.min():+.3f}. Across all {len(codes)} it "
      f"averages {bias_now.mean():+.4f} - it does not cancel.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
for lab, col in [("true > 0.8", WARN), ("all", INK), ("true < 0.4", "#38a169")]:
    ax.plot(grid, curves[lab], marker="o", ms=4, lw=1.8, color=col, label=lab)
ax.axhline(0, color=MUTED, lw=1)
ax.axhspan(-TOL, TOL, color=MUTED, alpha=0.12)
ax.text(60, TOL * 0.55, f"within the +/-{TOL} tolerance", fontsize=8.5, color=MUTED)
ax.set_xscale("log")
ax.set_xlabel("measurements per month")
ax.set_ylabel("average bias of the published score")
ax.set_title("Good countries are understated when measurements are few", fontsize=10.5)
ax.legend(fontsize=8.5, title="true score", title_fontsize=8)

ax = axes[1]
ax.scatter(np.maximum(n_med[~bad], 1), bias_now[~bad], s=18, alpha=0.55, color=ACCENT)
ax.scatter(np.maximum(n_med[bad], 1), bias_now[bad], s=30, alpha=0.85, color=WARN,
           label=f"bias exceeds the tolerance ({bad.sum()})")
ax.axhline(0, color=INK, lw=1)
ax.axhspan(-TOL, TOL, color=MUTED, alpha=0.12)
ax.set_xscale("log")
ax.set_xlabel("measurements collected per month")
ax.set_ylabel("bias of the published score")
ax.set_title("Which countries are affected today", fontsize=10.5)
ax.legend(fontsize=8.5, loc="lower right")
fig.tight_layout()
save(fig, "fig6_bias")
plt.show()

## 12. The Assumption Ledger

| | assumption | status | what it costs if wrong |
|---|---|---|---|
| **A1** | IQB flattens to ten weighted pass/fail tests | **Proved** — exact to 1e-16 against `IQBCalculator` (§2) | Everything. It is a fact about the config, not a modelling choice. |
| **A2** | The percentile index is a quality rank for all four metrics | **Verified** — 0 violations in 5,720 country-months (§3) | Latency and loss would be scored backwards. |
| **A3** | Measurements within a country-month are i.i.d. | **Known false, unquantifiable here** (§4, §11a) | Every requirement is a lower bound by an unknown factor. **The main open risk.** |
| **A4** | Nine percentile knots locate a threshold well enough | **Tested against 3 alternatives** (§10) | Aggregate: nothing. Per country: 22 of 245 are unresolved rather than large. |
| **A5** | The 24-month pooled distribution is the right target | **Priced** (§10) | Median requirement 563 → 706, P90 16,737 → 29,460. The largest live effect. |
| **A6** | The four metrics are independent | **Tested at the opposite extreme** (§10) | Nothing — one binding test sets the requirement, so dependence has nothing to act on. |
| **A7** | "Reliable" = within ±0.05, 95% of the time | **Priced** (§10) | τ=0.02 → 885; τ=0.10 → 380. A judgement about the index, not a statistical question. |
| **A8** | Sample quantile convention $r = \lceil qn \rceil$ | Standard; alternatives differ only at very small *n* | Negligible. |

---

## What I am asking you to check

1. **Is A3 fatal or survivable?** I can show the direction of the bound but not its size. Does
   "every number is a minimum" leave the conclusions usable, or does the clustering in §11a mean the
   requirements should be presented only as relative rankings rather than absolute counts?
2. **Is A5 the right target?** Pooling makes the margin estimable for small countries — that is the
   only reason it is there — but it changes what "true score" means. Should the whole analysis be
   monthly, and small countries simply declared unmeasurable?
3. **Is A4 acceptable, or a blocker?** The aggregate is untouched by it, but 22 countries have
   requirements that nine knots cannot determine. Is reporting those as "unresolved" honest enough,
   or does it undermine publishing per-country numbers at all?
4. **Is ±0.05 the right bar,** given that 6 of the 10 tests individually weigh more than it?
5. **Does the bias in §11b change what should be published?** My reading is that it argues for
   bias-correcting or suppressing rather than for wider error bars. I would like that challenged.

---

## If the assumptions hold, the recommendations are

1. **Gate per (country, percentile level), not on a global n** — every requirement is computable in
   closed form from data already on hand.
2. **Ask the pipeline for the ten pass rates.** Because IQB at level *q* is
   $\sum_k w_k \mathbf{1}\{q > 1-\pi_k\}$, those ten numbers are a *sufficient statistic for the
   entire IQB-versus-percentile curve*. They cost one `COUNT` per threshold and they eliminate A4
   entirely, fix the 12% of thresholds that fall outside the reported range, and make A3 checkable
   for the first time.
3. **Correct or flag the bias** before anyone ranks countries against each other.
4. **Publish a band, not a point,** for the countries sitting on a threshold.
5. **Stop increasing volume beyond ~10³ measurements a month** — above that, sampling error accounts for
   well under 1% of the movement actually observed.

Full analysis: `03-sample_size_reliability.ipynb`. Method write-up: `METHODOLOGY.md`.
Per-country numbers: `analysis_outputs/03_sample_size_requirements.csv`.
Figures from this notebook: `analysis_outputs/review_figures/`.


In [ ]:
print("figures written for lifting into slides or a write-up:\n")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"   {p}   ({p.stat().st_size / 1024:,.0f} KB)")